# LLW neighbourhood evidence across vaccination repertoires

Which clonotypes receive LLWNGPMAV reference support when every eligible TRB receptor is retained? Rank by LLW evidence, then inspect abundance. All donors have the author-declared HLA-A*02 restriction.

| Input | Parameter | Output |
| --- | --- | --- |
| All local AIRR tables and `metadata.txt` | `--dataset` | Complete replicate denominators and exclusion counts |
| Fixed VDJdb reference | `--reference` | Relative LLW score, radius support and exact-junction support |
| CPU and RAM budgets | `--threads 4 --memory-gb 24` | Measured runtime/RAM; explicit failure above the ceiling |

Install `vdjmatch[bench,notebook]` and `ipykernel`. Keep the complete examples directory together. Set `AIRR_YFV19` and `VDJDB_REFERENCE` to your raw mirror and reference. Original identifiers and detailed tables stay private.

**Execution status:** full-cohort workflow draft. The four-worker run has not
met the 30-minute budget; no completed cohort results are presented here.

## Run from the terminal

```sh
python examples/vaccination.py --dataset "$AIRR_YFV19" \
  --reference "$VDJDB_REFERENCE" --threads 4 --memory-gb 24 \
  --output NEW_PRIVATE_DIRECTORY
```

This command rebuilds raw-input tables, invokes the installed calculation CLI and exports TSV/SVG/PDF/PNG results. No cell editing or abundance cutoff is required. The notebook invokes the same command in a fresh directory.

Four CPU workers process one native search batch. Identical junction/V/J keys shared between replicates are searched once, then original replicate read counts are restored.

In [ ]:
# Set budgets before imports and report the actual software and host environment.
import os, sys, tempfile, platform
from pathlib import Path
from importlib.metadata import version
for key in ("POLARS_MAX_THREADS", "OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "vdjmatch-matplotlib"))
import polars as pl
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(20)
import psutil
from IPython.display import Image, display, Markdown
examples = next(p for p in (Path.cwd() / "examples", Path.cwd()) if (p / "vaccination.py").is_file())
sys.path.insert(0, str(examples))
from profile_cli import profile
print("Python", platform.python_version(), platform.machine())
for package in ("vdjmatch", "vdjtools", "seqtree", "polars", "matplotlib", "psutil"):
    print(package, version(package))
print("Physical cores", psutil.cpu_count(logical=False), "logical CPUs", psutil.cpu_count(),
      "RAM (decimal GB)", round(psutil.virtual_memory().total / 1e9, 2))

## Preserve the complete population

The reviewed local mirror contains six donors with two day-zero technical replicates each. Metadata also names later time points; this lesson describes files actually present locally.

A clonotype is one supplied V/junction/J amino-acid key within a replicate. `junction_aa` is the producer-declared full junction interval. Valid amino-acid sequences with noncanonical anchor residues remain present and flagged; functionality is not reported. Read counts are summed within identical keys; missing J calls remain distinct and retained. Non-TRB rows, invalid junctions and missing V calls are counted explicitly.

In [ ]:
# Rebuild the complete workflow through the terminal runner, with a measured whole-tree RAM ceiling.
dataset = Path(os.environ.get("AIRR_YFV19", Path.home() / "hf/airr_yfv19"))
benchmark = Path(os.environ.get("AIRR_BENCHMARK", Path.home() / "hf/airr_benchmark"))
reference = Path(os.environ.get("VDJDB_REFERENCE", benchmark / "vdjdb/vdjdb-2026-06-11-ZENODO/vdjdb_full.txt.gz"))
output = Path(tempfile.mkdtemp(prefix="vdjmatch-vaccination-")) / "fresh"
command = [sys.executable, str(examples / "vaccination.py"), "--dataset", str(dataset),
           "--reference", str(reference), "--threads", "4", "--memory-gb", "24", "--output", str(output)]
whole_usage = profile(command, output.parent / "workflow.log", timeout=1800,
                      max_rss_bytes=32_000_000_000)
coverage = pl.read_csv(output / "coverage.tsv", separator="\t")
support = pl.read_csv(output / "support.tsv", separator="\t")

## Check retained rows and read counts

Every local donor and technical replicate appears in `coverage.tsv`. Eligible read counts provide the complete denominator; the analysis does not discard rare clonotypes. The displayed replicate numbers are neutral, and original mappings remain in private manifests.

In [ ]:
# Display complete population accounting without original donor or sample identifiers.
public_coverage = coverage.with_columns(pl.col("replicate").rank("dense").over("donor", "day").alias("replicate_number")).drop("replicate")
display(public_coverage.sort("donor", "day", "replicate_number"))
print("Raw rows", coverage["raw_rows"].sum(), "eligible rows", coverage["eligible_rows"].sum(),
      "eligible read counts", coverage["eligible_read_counts"].sum())

## Read the CLI evidence

The CLI retains exact junction evidence and uses ordinary all-valid VDJdb membership. `--targets-from-sample` builds the compatible LLW/A*02 reference union.

`kernel_score` is higher-is-better relative evidence against other-peptide database junctions. `status = matched` means an LLW neighbour exists inside the declared distance radius. Exact-junction support is a separate flag, ignoring V/J identity. Each output retains unsupported query dispositions.

In [ ]:
# Verify every replicate denominator and display its support counts.
assert coverage.height == support.height
assert support["clonotypes"].to_list() == coverage["eligible_clonotypes"].to_list()
assert support["read_counts"].to_list() == coverage["eligible_read_counts"].to_list()
public_support = support.with_columns(pl.col("replicate").rank("dense").over("donor", "day").alias("replicate_number")).drop("replicate")
display(public_support.sort("donor", "day", "replicate_number"))

## Rank neighbourhood evidence before abundance

Panel a admits complete score-tied groups from strongest to weakest LLW evidence. The horizontal axis counts clonotypes; the vertical axis counts reads. Both denominators include every eligible clonotype, including unsupported keys.

Panel b shows the fraction with an LLW radius neighbour in each technical replicate. Two replicate observations belong to one donor, not two independent donors.

In [ ]:
# Display the runner's fresh figure, generated from authoritative CLI tables.
display(Image(filename=str(output / "llw-neighbour-evidence.png")))

**Figure 1 | LLW reference evidence across complete vaccination repertoires.** **a**, Cumulative eligible TRB read fraction against cumulative clonotype fraction, ranked by decreasing LLW evidence. Colours identify donors; separate line styles identify technical replicates. Complete score ties enter together; unavailable scores enter last. **b**, Radius-supported clonotype fraction, with one point per technical replicate. Clonotypes are amino-acid V/junction/J keys; abundance measures reads. Exact reference evidence is retained.

## Inspect individual results privately

Sort `observations.tsv` by decreasing `kernel_score` to inspect strong reference-supported candidates and their replicate read counts. `llw.queries.tsv` explains unsupported cases; `llw.selection.tsv` retains each search key's score and exact-junction support.

`manifest.json` records raw/reference hashes, private identity mappings and the complete calculation command. The CLI manifest records actual geometry and software identities. Detailed raw receptor records stay private.

In [ ]:
# State the quantitative support range from the newly generated count table.
fraction = support["radius_supported_clonotypes"] / support["clonotypes"]
display(Markdown(f"Across **{support.height} repertoires**, LLW radius support spans "
    f"**{100 * fraction.min():.3f}–{100 * fraction.max():.3f}%** of eligible clonotypes. "
    "Every eligible clonotype remains available for evidence ranking; abundance does not define the input subset."))

## Runtime and RAM

The CLI measurement includes startup, loading, scoring and table export. The whole-run measurement also includes raw preparation, count restoration, summaries and plotting. Neither includes the notebook kernel. Summed RSS may count shared pages twice; 50-ms sampling may miss brief peaks.

In [ ]:
# Report CLI and complete-workflow costs separately and verify all expected artifacts.
print(pl.read_csv(output / "resources.tsv", separator="\t"))
print(pl.DataFrame([{"scope": "Preparation + CLI + export + figures", **whole_usage}]))
required = ["coverage.tsv", "support.tsv", "observations.tsv", "evidence-coverage.tsv", "resources.tsv",
            "manifest.json", "llw.queries.tsv", "llw.selection.tsv", "llw.manifest.json",
            "llw-neighbour-evidence.png", "llw-neighbour-evidence.svg", "llw-neighbour-evidence.pdf"]
assert all((output / name).is_file() for name in required)
print("Verified", len(required), "final artifacts")

## Interpretation and limits

The workflow ranks LLW reference-neighbour evidence across all locally available donors and eligible clonotypes. Read coverage and clonotype coverage are separate denominators.

- Radius/exact support and kernel evidence do not establish binding prevalence; no assay labels or matching final call calibration are supplied here.
- Technical replicates remain separate, and read abundance is not a cell count. Receptor functionality is unknown, including for the retained noncanonical-anchor rows.
- Day-zero repertoires cannot establish vaccine-induced expansion. That question requires longitudinal inputs and their recorded identities.

## Final artifacts

The fresh output directory contains complete coverage/support tables, per-clonotype evidence, evidence-ranked coverage curves, CLI manifests/logs, resource measurements and vector/raster figures. A new terminal invocation requires a new directory and rebuilds every calculation. Aggregate figures and counts can be reviewed without exposing original donor names, sample IDs or raw receptors.